In [ ]:
!pip install "numpy==1.26.4"
# Cài đặt SciPy và Sklearn phù hợp với NumPy 1.x
!pip install "scipy==1.11.4" "scikit-learn==1.3.2"
!pip install torchtext==0.6.0

In [ ]:
import numpy as np
import torchtext
import sklearn
from sklearn.metrics import precision_recall_fscore_support

print(f"NumPy: {np.__version__} (Chuẩn 1.x)")
print(f"Torchtext: {torchtext.__version__} (Chuẩn 0.6.0)")
print("Sklearn: Import thành công!")

In [ ]:
import argparse

class KaggleConfig:
    def __init__(self):
        self.vocab_size = 100000
        self.batch_size = 32 # Giảm xuống 32 để tránh lỗi bộ nhớ GPU
        self.hidden_dim = 128
        self.max_node = 300
        self.max_token = 20
        self.learning_rate = 0.001
        self.epoch = 10
        self.cpu = False
        self.gpu = 0
        self.save_model = True
        self.train_path = '/kaggle/input/datasets/tranminhtuan204/original-cfgnn-data/train_api.csv'
        self.test_path = '/kaggle/input/datasets/tranminhtuan204/original-cfgnn-data/test_api.csv'
        self.valid_path = '/kaggle/input/datasets/tranminhtuan204/original-cfgnn-data/valid_api.csv'

opt = KaggleConfig()

In [ ]:
import torch
import pandas as pd
try:
    from torchtext import data
except ImportError:
    from torchtext.legacy import data

def read_data(data_path, fields):
    print(f"Đang đọc: {data_path}")
    # Đọc theo chunksize 
    csv_data = pd.read_csv(data_path, chunksize=5000)
    all_examples = []
    for chunk in csv_data:
        examples = chunk.apply(lambda r: data.Example.fromlist([
            eval(r[0]), eval(r[1]), eval(r[2]), eval(r[3]), r[4]
        ], fields), axis=1)
        all_examples.extend(list(examples))
    return all_examples

def get_iterators(args, device):
    # Định nghĩa các trường dữ liệu theo chuẩn Legacy
    TEXT = data.Field(tokenize=lambda x: x.split()[:args.max_token])
    NODE = data.NestedField(TEXT, preprocessing=lambda x: x[:args.max_node], include_lengths=True)
    ROW = data.Field(pad_token=1.0, use_vocab=False, 
                     preprocessing=lambda x: [1, 1] if any(i > args.max_node for i in x) else x)
    EDGE = data.NestedField(ROW)
    TYPE = data.Field(use_vocab=False, preprocessing=lambda x: x[:args.max_node], 
                      pad_token=0, batch_first=True)
    LABEL = data.Field(sequential=False, use_vocab=False)
    
    fields = [("nodes", NODE), ("f_edges", EDGE), ("b_edges", EDGE), ("type", TYPE), ("label", LABEL)]

    print("Bắt đầu tạo Dataset...")
    train_ds = data.Dataset(read_data(args.train_path, fields), fields)
    valid_ds = data.Dataset(read_data(args.valid_path, fields), fields)
    test_ds = data.Dataset(read_data(args.test_path, fields), fields)

    print("Đang xây dựng từ vựng (Vocabulary)...")
    NODE.build_vocab(train_ds, max_size=args.vocab_size)

    train_iter, valid_iter, test_iter = data.Iterator.splits(
        (train_ds, valid_ds, test_ds),
        batch_size=args.batch_size,
        device=device,
        sort=False, repeat=False
    )
    return train_iter, valid_iter, test_iter

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support
import os
import warnings


warnings.simplefilter("ignore")

# model 

class CFGNN(nn.Module):
    def __init__(self, opt):
        super(CFGNN, self).__init__()
        self.hidden_dim = opt.hidden_dim
        self.embedding = nn.Embedding(opt.vocab_size+2, opt.hidden_dim, padding_idx=1)
        self.node_lstm = nn.LSTM(self.hidden_dim, self.hidden_dim//2, bidirectional=True, batch_first=True)
        self.gate = Gate(self.hidden_dim, self.hidden_dim)
        self.back_gate = Gate(self.hidden_dim, self.hidden_dim)
        self.attention = GlobalAttention(self.hidden_dim*2, attn_type='mlp')
        self.fc_output = nn.Linear(self.hidden_dim*4, 1)

    def forward(self, x, edges, node_lens=None, token_lens=None, target=None):
        f_edges, b_edges = edges
        batch_size, num_node, num_token = x.size(0), x.size(1), x.size(2)
        
        x = self.embedding(x)
        if token_lens is not None:
            x = x.view(batch_size*num_node, num_token, -1)
            x, _ = self.node_lstm(x)
            x = x.view(batch_size, num_node, num_token, -1)
            x = self.average_pooling(x, token_lens)
        else:
            x = torch.mean(x, dim=2)

        h = torch.zeros(x.size(), device=x.device)
        c = torch.zeros(x.size(), device=x.device)
        f_matrix = self.convert_to_matrix(batch_size, num_node, f_edges)

        for i in range(num_node):
            x_cur = x[:, i, :]
            f_i = f_matrix[:, i, :].unsqueeze(1)
            h_last, c_last = f_i.bmm(h), f_i.bmm(c)
            h_i, c_i = self.gate(x_cur, h_last.squeeze(1), c_last.squeeze(1))
            h[:, i, :], c[:, i, :] = h_i, c_i

        # Backward pass logic
        h_b = torch.zeros(x.size(), device=x.device)
        c_b = torch.zeros(x.size(), device=x.device)
        b_matrix = self.convert_to_matrix(batch_size, num_node, f_edges).transpose(1, 2)
        
        for i in reversed(range(num_node)):
            x_cur = x[:, i, :]
            b_i = b_matrix[:, i, :].unsqueeze(1)
            h_hat, c_hat = b_i.bmm(h_b), b_i.bmm(c_b)
            h_b[:, i, :], c_b[:, i, :] = self.back_gate(x_cur, h_hat.squeeze(1), c_hat.squeeze(1))

        h_combined = torch.cat([h, h_b], dim=2)
        
        if target is not None:
            output, _ = self.attention(h_combined, node_lens, target)
        else:
            output = torch.mean(h_combined, dim=1)
            
        return torch.sigmoid(self.fc_output(output))

    @staticmethod
    def average_pooling(data, input_lens):
        B, N, T, H = data.size()
        idx = torch.arange(T, device=data.device).view(1, 1, T).expand(B, N, T)
        mask = (idx < input_lens.unsqueeze(2)).float().unsqueeze(3)
        return (data * mask).sum(2) / (input_lens.unsqueeze(2).float() + 1e-9)

    @staticmethod
    def convert_to_matrix(batch_size, max_num, m):
        matrix = torch.zeros((batch_size, max_num, max_num), device=m.device)
        m_shifted = m - 1
        # Tránh lỗi index -1
        m_shifted = torch.clamp(m_shifted, min=0)
        
        b_idx = torch.arange(batch_size, device=m.device).view(-1, 1).expand_as(m_shifted[:,:,0])
        matrix[b_idx, m_shifted[:, :, 1], m_shifted[:, :, 0]] = 1
        matrix[:, 0, 0] = 0
        return matrix
    
class Gate(nn.Module):
    def __init__(self, in_dim, mem_dim):
        super(Gate, self).__init__()
        self.in_dim = in_dim
        self.mem_dim = mem_dim
        self.ax = nn.Linear(self.in_dim, 3 * self.mem_dim)
        self.ah = nn.Linear(self.mem_dim, 3 * self.mem_dim)
        self.fx = nn.Linear(self.in_dim, self.mem_dim)
        self.fh = nn.Linear(self.mem_dim, self.mem_dim)

    def forward(self, inputs, last_h, pred_c):
        iou = self.ax(inputs) + self.ah(last_h)
        i, o, u = torch.split(iou, iou.size(1) // 3, dim=1)
        # Sửa F.sigmoid thành torch.sigmoid để tương thích PyTorch mới
        i, o, u = torch.sigmoid(i), torch.sigmoid(o), torch.tanh(u)

        f = torch.sigmoid(self.fh(last_h) + self.fx(inputs))
        fc = torch.mul(f, pred_c)

        c = torch.mul(i, u) + fc
        h = torch.mul(o, torch.tanh(c))
        return h, c

class GlobalAttention(nn.Module):
    def __init__(self, dim, coverage=False, attn_type="mlp", attn_func="softmax"):
        super(GlobalAttention, self).__init__()
        self.dim = dim
        self.attn_type = attn_type
        self.attn_func = attn_func
        self.annotation = nn.Sequential(nn.Embedding(2, 1), nn.Linear(1, 1))

        if self.attn_type == "mlp":
            self.linear_context = nn.Linear(dim, dim, bias=False)
            self.linear_query = nn.Linear(dim, dim, bias=True)
            self.v = nn.Linear(dim, 1, bias=False)

    def score(self, h_t, h_s):
        batch_size, src_len, _ = h_s.size()
        if self.attn_type == "mlp":
            target_features = self.linear_query(h_t) 
            source_features = self.linear_context(h_s) 
            combined_features = torch.tanh(target_features + source_features)
            return self.v(combined_features).transpose(1, 2)
        return torch.bmm(h_t, h_s.transpose(1, 2))

    def forward(self, memory_bank, memory_lengths=None, targets=None):
        batch, source_l, dim = memory_bank.size()
        source_query = torch.max(memory_bank, dim=1)[0].unsqueeze(1)
        align = self.score(source_query, memory_bank)
        
        if targets is not None:
            ann = self.annotation(targets).transpose(1, 2)
            align = align * ann

        if memory_lengths is not None:
            # Gọi hàm sequence_mask đã được định nghĩa bên dưới
            mask = self.sequence_mask(memory_lengths, max_len=source_l).unsqueeze(1)
            align.masked_fill_(~mask, -float('inf'))

        align_vectors = F.softmax(align.view(batch, source_l), dim=-1).unsqueeze(1)
        c = torch.bmm(align_vectors, memory_bank)
        concat_c = torch.cat([c, source_query], 2).squeeze(1) 
        return concat_c, align_vectors.squeeze(1)

    # THÊM ĐOẠN NÀY VÀO BÊN TRONG LỚP GlobalAttention
    @staticmethod
    def sequence_mask(lengths, max_len=None):
        """
        Tạo mặt nạ boolean từ độ dài chuỗi.
        """
        batch_size = lengths.numel()
        max_len = max_len or lengths.max()
        return (torch.arange(0, max_len, device=lengths.device)
                .type_as(lengths)
                .repeat(batch_size, 1)
                .lt(lengths.unsqueeze(1)))

# Train + Val

def train(opt, train_iter, valid_iter, device):
    net = CFGNN(opt).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(net.parameters(), lr=opt.learning_rate)
    
    print(f'Bắt đầu huấn luyện trên thiết bị: {device}')
    
    for i in range(opt.epoch):
        total_loss, total_acc = [], []
        net.train()
        
        for batch in tqdm(train_iter, desc=f"Epoch {i}"):
            x, m, t = batch.nodes, (batch.f_edges, batch.b_edges), batch.label.float()
            
            if isinstance(x, tuple):
                pred = net(x[0], m, node_lens=x[1], token_lens=None, target=batch.type)
            else:
                pred = net(x, m)
                
            pred = pred.squeeze()
            loss = criterion(pred, t)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss.append(loss.item())
            total_acc.append(((pred > 0.5).float() == t).float().mean().item())

        # Validation
        net.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for batch in valid_iter:
                x, m, t = batch.nodes, (batch.f_edges, batch.b_edges), batch.label.float()
                pred = net(x[0], m, x[1], target=batch.type) if isinstance(x, tuple) else net(x, m)
                pred = pred.squeeze()
                y_true.extend(t.tolist())
                y_pred.extend((pred > 0.5).float().tolist())

        metrics = precision_recall_fscore_support(y_true, y_pred, average='binary')
        print(f"Epoch {i} | Loss: {sum(total_loss)/len(total_loss):.4f} | Acc: {sum(total_acc)/len(total_acc):.4f}")
        print(f"Val Precision: {metrics[0]:.4f} | Recall: {metrics[1]:.4f} | F1: {metrics[2]:.4f}")

        os.makedirs('checkpoints/', exist_ok=True)
        torch.save(net.state_dict(), f'checkpoints/epoch-{i}.pt')

    return net

def main():
    global opt
    
    # Thiết lập device
    device = torch.device(f"cuda:{opt.gpu}" if torch.cuda.is_available() and not opt.cpu else "cpu")
    
    print("Đang tải dữ liệu và khởi tạo Iterator...")

    train_iter, valid_iter, test_iter = get_iterators(opt, device)
    
    print(f"Bắt đầu huấn luyện trên: {device}")
    train(opt, train_iter, valid_iter, device)

if __name__ == '__main__':
    main()

In [ ]:
def evaluate_on_test(opt, checkpoint_path):
    device = torch.device(f"cuda:{opt.gpu}" if torch.cuda.is_available() and not opt.cpu else "cpu")
    print(f"Thiết bị đang sử dụng: {device}")

    print("Đang nạp dữ liệu tập Test...")
    _, _, test_it = get_iterators(opt, device)
    
    net = CFGNN(opt).to(device)
    print(f"Đang tải trọng số từ: {checkpoint_path}")
    
    net.load_state_dict(torch.load(checkpoint_path, map_location=device))
    net.eval()
    
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for batch in tqdm(test_it, desc="Testing"):
            x, m, t = batch.nodes, (batch.f_edges, batch.b_edges), batch.label.float()
            
            if isinstance(x, tuple):
                pred = net(x[0], m, node_lens=x[1], target=batch.type)
            else:
                pred = net(x, m)
            
            pred = pred.squeeze()
            
            # Đảm bảo pred và t luôn là tensor có ít nhất 1 chiều để tránh lỗi scalar
            if pred.dim() == 0:
                pred = pred.unsqueeze(0)
            if t.dim() == 0:
                t = t.unsqueeze(0)

            y_true.extend(t.cpu().numpy().tolist())
            y_pred.extend((pred > 0.5).float().cpu().numpy().tolist())
            
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    accuracy = (torch.tensor(y_pred) == torch.tensor(y_true)).float().mean().item()
    
    print("\n" + "="*40)
    print("KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST")
    print("="*40)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {p:.4f}")
    print(f"Recall:    {r:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print("="*40)

evaluate_on_test(opt, 'checkpoints/epoch-9.pt')